In [15]:
import os
import time
import pandas as pd
from PIL import Image
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv")
sample_df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv")

In [3]:
BASE_TRAIN_DIR = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train"
BASE_TEST_DIR = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test"

In [4]:
train_df['class'].unique()

array([0, 1])

In [5]:
sample_df['answer'] = 0

In [6]:
train_df

,image_name,class
0,ChokChai4_img_13-7956791_100-6031267_a187-2159...,0
1,ChokChai4_img_13-7961753_100-6031881_a185-9785...,0
2,ChokChai4_img_13-7969811_100-5906061_a180-5812...,0
3,ChokChai4_img_13-7970811_100-5906071_a180-5812...,0
4,ChokChai4_img_13-7971811_100-5906081_a180-5812...,0
...,...,...
2948,MrtSutthisan_img_13-7986837_100-5730268_a260_s...,1
2949,MrtSutthisan_img_13-7990804_100-5729745_a170_s...,1
2950,MrtSutthisan_img_13-7991266_100-5728237_a255_s...,1
2951,MrtSutthisan_img_13-7992342_100-5710656_a55_s9...,1


In [7]:
train_df['image_name'][0]

'ChokChai4_img_13-7956791_100-6031267_a187-21598902886774_s90_y0_f90_1.jpg'

In [8]:
sample_df['id'][0]

'e4b420b0'

In [9]:
sample_df

,id,answer
0,e4b420b0,0
1,23efa479,0
2,1f0f2402,0
3,8a60480c,0
4,11f20127,0
...,...,...
1545,ed44ac4d,0
1546,a2067109,0
1547,1481178a,0
1548,340a3e9d,0


In [11]:
print("Torch path:", torch.__file__)
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print(torch.cuda.device_count())

Torch path: /usr/local/lib/python3.12/dist-packages/torch/__init__.py
Torch version: 2.10.0+cu128
CUDA version: 12.8
GPU: Tesla T4
2


In [12]:
def set_seed(seed=42):
    # Python
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch (CPU)
    torch.manual_seed(seed)

    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call this at the very top of your notebook
set_seed(42)

In [14]:
# =========================
# EfficientNet Ensemble (B0 + B2 + B4) - Full Pipeline
# =========================

set_seed(42)

# =========================
# Config
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 8
lr = 1e-4
batch_size = 32

# =========================
# Dataset
# =========================
class HouseDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.is_test:
            img_id = self.df.iloc[idx, 0]
            img_name = img_id + ".jpg"
        else:
            img_name = self.df.iloc[idx, 0]

        img_path = os.path.join(self.img_dir, img_name)

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Missing file: {img_path}")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, img_id
        else:
            label = self.df.iloc[idx, 1]
            return image, label

# =========================
# Transforms
# =========================
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# =========================
# Split
# =========================
train_split, val_split = train_test_split(
    train_df, test_size=0.2, stratify=train_df["class"], random_state=42
)

train_loader = DataLoader(HouseDataset(train_split, BASE_TRAIN_DIR, train_tfms),
                          batch_size=batch_size, shuffle=True, num_workers=2)

val_loader = DataLoader(HouseDataset(val_split, BASE_TRAIN_DIR, val_tfms),
                        batch_size=batch_size, shuffle=False, num_workers=2)

test_loader = DataLoader(HouseDataset(sample_df, BASE_TEST_DIR, val_tfms, is_test=True),
                         batch_size=batch_size, shuffle=False, num_workers=2)

num_classes = train_df["class"].nunique()

# =========================
# Model Factory
# =========================
def create_model(name):
    if name == "b0":
        model = models.efficientnet_b0(pretrained=True)
    elif name == "b2":
        model = models.efficientnet_b2(pretrained=True)
    elif name == "b4":
        model = models.efficientnet_b4(pretrained=True)

    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

# =========================
# Train Function
# =========================
def train_model(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_acc = 0
        count = 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)

                val_acc += (preds == labels).sum().item()
                count += labels.size(0)

        val_acc /= count
        print(f"{model.__class__.__name__} | Epoch {epoch+1} | Loss={train_loss/len(train_loader):.4f} | ValAcc={val_acc:.4f}")

    return model

# =========================
# Train Models
# =========================
model_b0 = train_model(create_model("b0"))
model_b2 = train_model(create_model("b2"))
model_b4 = train_model(create_model("b4"))

models_list = [model_b0, model_b2, model_b4]

# =========================
# Ensemble Inference
# =========================
for m in models_list:
    m.eval()

predictions = []

with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device)

        probs_sum = 0

        for m in models_list:
            outputs = m(imgs)
            probs = F.softmax(outputs, dim=1)
            probs_sum += probs

        probs_avg = probs_sum / len(models_list)
        preds = torch.argmax(probs_avg, dim=1).cpu().numpy()

        for i, p in zip(ids, preds):
            predictions.append((i, p))

# =========================
# Save CSV
# =========================
submission = pd.DataFrame(predictions, columns=["id", "answer"])

timestamp = time.strftime("%Y%m%d-%H%M%S")
filename = f"submission_{timestamp}.csv"

submission.to_csv(filename, index=False)

print("Saved:", filename)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EfficientNet | Epoch 1 | Loss=0.3732 | ValAcc=0.9408
EfficientNet | Epoch 2 | Loss=0.1619 | ValAcc=0.9475
EfficientNet | Epoch 3 | Loss=0.0779 | ValAcc=0.9611
EfficientNet | Epoch 4 | Loss=0.0537 | ValAcc=0.9442
EfficientNet | Epoch 5 | Loss=0.0355 | ValAcc=0.9475
EfficientNet | Epoch 6 | Loss=0.0315 | ValAcc=0.9577
EfficientNet | Epoch 7 | Loss=0.0253 | ValAcc=0.9577
EfficientNet | Epoch 8 | Loss=0.0178 | ValAcc=0.9459


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B2_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EfficientNet | Epoch 1 | Loss=0.3700 | ValAcc=0.9509
EfficientNet | Epoch 2 | Loss=0.1471 | ValAcc=0.9543
EfficientNet | Epoch 3 | Loss=0.0711 | ValAcc=0.9628
EfficientNet | Epoch 4 | Loss=0.0499 | ValAcc=0.9594
EfficientNet | Epoch 5 | Loss=0.0363 | ValAcc=0.9611
EfficientNet | Epoch 6 | Loss=0.0277 | ValAcc=0.9509
EfficientNet | Epoch 7 | Loss=0.0287 | ValAcc=0.9594
EfficientNet | Epoch 8 | Loss=0.0180 | ValAcc=0.9475


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B4_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B4_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EfficientNet | Epoch 1 | Loss=0.5868 | ValAcc=0.8832
EfficientNet | Epoch 2 | Loss=0.2969 | ValAcc=0.9272
EfficientNet | Epoch 3 | Loss=0.1866 | ValAcc=0.9475
EfficientNet | Epoch 4 | Loss=0.1341 | ValAcc=0.9408
EfficientNet | Epoch 5 | Loss=0.1023 | ValAcc=0.9475
EfficientNet | Epoch 6 | Loss=0.0710 | ValAcc=0.9475
EfficientNet | Epoch 7 | Loss=0.0705 | ValAcc=0.9560
EfficientNet | Epoch 8 | Loss=0.0420 | ValAcc=0.9560


NameError: name 'F' is not defined